# Step 4: Direction Analysis — Steering vs Assistant Axis

Investigate the geometric relationship between trait steering directions and
the assistant axis itself.

Questions:
- Are steering vectors aligned with, orthogonal to, or independent of the axis?
- Does alignment vary by trait? (e.g. sycophancy might align with the axis)
- Does the base model's steering vector relate differently to the axis than
  axis-persona vectors do?

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from persona_steering.config import VECTORS_DIR, ACTIVATIONS_DIR, Trait
from persona_steering.personas import load_all_personas, load_axis_personas
from persona_steering.analysis import compare_steering_vs_interpersona, decompose_shared_specific
from persona_steering.utils import load_pickle, ensure_output_dirs, cosine_similarity

ensure_output_dirs()
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
all_personas = load_all_personas()
axis_personas = load_axis_personas()
axis_slugs = [p.slug for p in axis_personas]
traits = list(Trait)

In [ ]:
# Compute the assistant axis direction from steering vectors.
# Approximate: direction from deep_roleplay mean to full_assistant mean
# across all traits at the mid layer.
sample_trait = traits[0]
sample_layers = sorted(all_vectors[axis_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]

# Mean vector per axis persona (across traits)
persona_means = {}
for slug in axis_slugs:
    vecs = [all_vectors[slug][t][mid_layer].vector for t in traits
            if t in all_vectors.get(slug, {}) and mid_layer in all_vectors[slug].get(t, {})]
    if vecs:
        persona_means[slug] = torch.stack(vecs).mean(dim=0)

# The assistant axis: from most anti-assistant to most assistant
anti_slug = axis_slugs[0]   # deep_roleplay
asst_slug = axis_slugs[-1]  # full_assistant
assistant_axis = persona_means[asst_slug] - persona_means[anti_slug]
print(f"Assistant axis magnitude: {assistant_axis.norm():.4f}")
print(f"Direction: {anti_slug} -> {asst_slug}")

In [ ]:
# Compare each trait's steering direction against the assistant axis
results = []
all_slugs = [p.slug for p in all_personas]

for trait in traits:
    for slug in all_slugs:
        vec = all_vectors.get(slug, {}).get(trait, {}).get(mid_layer)
        if vec is None:
            continue
        comp = compare_steering_vs_interpersona(vec, assistant_axis)
        persona = next(p for p in all_personas if p.slug == slug)
        pos_str = f"{persona.position:+.1f}" if persona.position is not None else "base"
        comp["trait"] = trait.value
        comp["persona"] = slug
        comp["position"] = pos_str
        results.append(comp)

df = pd.DataFrame(results)
print("Mean alignment by trait:")
print(df.groupby("trait")["cosine_similarity"].mean().sort_values())

In [ ]:
# Heatmap: steering-axis alignment by trait × persona
fig, ax = plt.subplots(figsize=(10, 5))
pivot = df.pivot_table(values="cosine_similarity", index="trait", columns="persona")
# Reorder columns by axis position
col_order = [s for s in axis_slugs if s in pivot.columns]
base_cols = [s for s in pivot.columns if s not in axis_slugs]
pivot = pivot[base_cols + col_order]

sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Trait Steering Vector Alignment with Assistant Axis")
plt.tight_layout()
plt.savefig("../outputs/figures/steering_axis_alignment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Alignment ratio by trait — what fraction lies along the axis?
fig, ax = plt.subplots(figsize=(8, 5))

pivot_ratio = df.pivot_table(values="alignment_ratio", index="trait", columns="position")
pivot_ratio.plot(kind="bar", ax=ax)
ax.set_ylabel("|proj onto axis| / |steering vec|")
ax.set_title("Fraction of Steering Vector Aligned with Assistant Axis")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../outputs/figures/alignment_ratios.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-trait shared direction analysis across axis personas
print("\nShared vs persona-specific decomposition (axis personas only):")
for trait in traits:
    vecs = {slug: all_vectors[slug][trait][mid_layer] for slug in axis_slugs
            if trait in all_vectors.get(slug, {}) and mid_layer in all_vectors[slug].get(trait, {})}
    if len(vecs) >= 2:
        decomp = decompose_shared_specific(vecs)
        # How aligned is the shared direction with the assistant axis?
        shared_axis_cos = cosine_similarity(decomp.shared_direction, assistant_axis)
        print(f"\n  {trait.value}:")
        print(f"    variance_explained by shared = {decomp.variance_explained:.4f}")
        print(f"    shared direction ↔ assistant axis cos = {shared_axis_cos:.4f}")